# EchoFactory — PUMP: echofac9 (STgram-MFN Optimized V3)

## Penyempurnaan dari Versi Sebelumnya:
1. **RAM Preloading (Super Cepat)**: Preload semua audio ke RAM di awal (~2-3 detik/epoch, total ~4-5 menit training).
2. **Dual Spectro-Temporal Features (STgram)**: Branch 0 (Log-Mel) + Branch 1 (Linear STFT High-Res).
3. **SpecAugment Regularization**: Time Masking + Frequency Masking untuk mencegah overfitting dini.
4. **Per-ID Anomaly Scoring (DCASE/MIMII Standard)**:
   - Evaluasi dilakukan berbasis ID mesin target ($W_{ID}$ & KNN Cosine per-ID).
   - Menghitung AUC per-ID dan Overall Ensemble AUC (Target: **>94%**).
5. **Export ONNX & JSON Bebas Error**: Auto-install dependensi `onnx` & `onnxscript` dan export opset 17.


In [ ]:
# =====================================================================
# CELL 1: SETUP & KONFIGURASI
# =====================================================================
import os, gc, glob, json, math, time, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

# =====================================================================
# KONFIGURASI
# =====================================================================
DATASET_ROOT = '/kaggle/input/datasets/bisheshgiri/mimii-dataset'
OUT_DIR      = '/kaggle/working'
MACHINE_TYPE = 'pump'
TARGET_SNR   = '0_dB'

SR        = 16000
AUDIO_LEN = SR * 10  # 160000 samples = 10 detik

# Feature Extraction Params
N_MELS    = 128
N_FFT_MEL = 1024
HOP_MEL   = 512

N_FFT_TG  = 512
HOP_TG    = 256
N_BINS_TG = 128

# Model & Training Hyperparameters
EMBED_DIM  = 128
ARC_S      = 30.0
ARC_M      = 0.5
BATCH_SIZE = 64
EPOCHS     = 100
LR         = 5e-4
WARMUP_EP  = 15
WEIGHT_DECAY = 1e-3

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Machine: {MACHINE_TYPE.upper()} | SNR: {TARGET_SNR}')
print(f'Audio: SR={SR}, len={AUDIO_LEN} ({AUDIO_LEN/SR}s)')
print(f'ArcFace: s={ARC_S}, m={ARC_M} | Epochs={EPOCHS} | BS={BATCH_SIZE}')


In [ ]:
# =====================================================================
# CELL 2: SPECAUGMENT (Mencegah Overfitting Dini)
# =====================================================================
class SpecAugment(nn.Module):
    """
    Frequency & Time Masking untuk regularisasi spektrogram.
    Mencegah model hanya menghafal background noise statis.
    """
    def __init__(self, freq_mask=12, time_mask=16):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask

    def forward(self, x):
        if not self.training:
            return x
        B, C, F_dim, T_dim = x.shape
        out = x.clone()
        for b in range(B):
            # Freq mask
            f_len = torch.randint(0, self.freq_mask + 1, (1,)).item()
            if f_len > 0 and F_dim > f_len:
                f_0 = torch.randint(0, F_dim - f_len, (1,)).item()
                out[b, :, f_0:f_0+f_len, :] = 0
            # Time mask
            t_len = torch.randint(0, self.time_mask + 1, (1,)).item()
            if t_len > 0 and T_dim > t_len:
                t_0 = torch.randint(0, T_dim - t_len, (1,)).item()
                out[b, :, :, t_0:t_0+t_len] = 0
        return out

print('SpecAugment module initialized.')


In [ ]:
# =====================================================================
# CELL 3: MODEL — Dual-Branch MobileFaceNet + ArcFace
# =====================================================================
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc), nn.PReLU(oc)
        )
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )
    def forward(self, x): return self.net(x)

class MobileFaceNetEncoder(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.aug = SpecAugment(freq_mask=12, time_mask=16)
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2),
            DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2),
            DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2),
            DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed)
        )
    def forward(self, x):
        x = self.aug(x)
        return self.head(self.enc(x))

class ArcFaceLoss(nn.Module):
    def __init__(self, ed, nc, s=30.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, feat, labels):
        cos = F.normalize(feat, dim=1) @ F.normalize(self.W, dim=1).T
        sin = (1.0 - cos.pow(2)).clamp(1e-9).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        oh  = F.one_hot(labels, cos.shape[1]).float()
        return F.cross_entropy((oh * phi + (1.0 - oh) * cos) * self.s, labels)

    @torch.no_grad()
    def class_weights(self):
        return F.normalize(self.W, dim=1)

class STgramMFN_v3(nn.Module):
    def __init__(self, n_classes, ed=128):
        super().__init__()
        self.mel_encoder   = MobileFaceNetEncoder(ed)
        self.tgram_encoder = MobileFaceNetEncoder(ed)
        self.fuse = nn.Sequential(
            nn.Linear(ed * 2, ed),
            nn.BatchNorm1d(ed),
            nn.PReLU(ed)
        )
        self.arc = ArcFaceLoss(ed, n_classes, s=ARC_S, m=ARC_M)

    def forward(self, mel, tg, labels=None):
        f_mel = self.mel_encoder(mel)
        f_tg  = self.tgram_encoder(tg)
        feat  = F.normalize(self.fuse(torch.cat([f_mel, f_tg], dim=1)), dim=1)

        if labels is not None:
            return feat, self.arc(feat, labels)
        return feat

# Verifikasi
tmp = STgramMFN_v3(n_classes=4, ed=EMBED_DIM)
n_p = sum(p.numel() for p in tmp.parameters() if p.requires_grad)
print(f'STgramMFN_v3 | Params: {n_p:,} ({n_p/1e6:.2f}M)')
with torch.no_grad():
    f, l = tmp(torch.randn(2, 1, 128, 128), torch.randn(2, 1, 128, 128), torch.tensor([0, 1]))
print(f'Output: feat={f.shape} (norm≈{f.norm(dim=1).mean():.2f}), loss={l.item():.4f}')
del tmp, f, l


In [ ]:
# =====================================================================
# CELL 4: DATASET — IN-MEMORY PRELOAD (Super Fast ~2-3s/epoch)
# =====================================================================
class MIMIIDataset_STgram_RAM(Dataset):
    """
    Preload Log-Mel & Tgram (STFT Linear) ke RAM sekali saja.
    Training berjalan 100% dari RAM tanpa bottleneck I/O disk.
    """
    def __init__(self, root, machine, snr, cond,
                 sr=16000, audio_len=160000):
        self.sr, self.audio_len = sr, audio_len

        mp = os.path.join(root, f'{snr}_{machine}', machine)
        if not os.path.exists(mp):
            raise FileNotFoundError(f'Path tidak ada: {mp}')

        ids = sorted(os.listdir(mp))
        self.id2label = {mid: i for i, mid in enumerate(ids)}
        self.n_classes = len(ids)
        print(f'Machine IDs: {self.id2label}')

        file_list = []
        for mid, lbl in self.id2label.items():
            cp = os.path.join(mp, mid, cond)
            if not os.path.exists(cp):
                print(f'  SKIP: {cp}'); continue
            files = sorted(glob.glob(os.path.join(cp, '*.wav')))
            file_list.extend([(f, lbl, mid) for f in files])
            print(f'  {mid}/{cond}: {len(files)} files')

        print(f'\n[Preloading {len(file_list)} files ({cond}) ke RAM...]')
        self.mels   = []
        self.tgrams = []
        self.labels = []
        self.mids   = []

        for fpath, lbl, mid in tqdm(file_list, desc=f'Preload {cond}'):
            wav, _ = sf.read(fpath, dtype='float32')
            if wav.ndim > 1:
                wav = wav.mean(axis=1)  # downmix mono
            if len(wav) >= self.audio_len:
                wav = wav[:self.audio_len]
            else:
                wav = np.pad(wav, (0, self.audio_len - len(wav)))

            # Branch 0: Log-Mel Spectrogram
            mel = librosa.feature.melspectrogram(
                y=wav, sr=self.sr, n_mels=N_MELS,
                n_fft=N_FFT_MEL, hop_length=HOP_MEL
            )
            mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)

            # Branch 1: High-Res Linear STFT (T-gram)
            stft = np.abs(librosa.stft(y=wav, n_fft=N_FFT_TG, hop_length=HOP_TG))
            tg = stft[:N_BINS_TG, :]  # (128, T)
            tg_db = librosa.amplitude_to_db(tg, ref=np.max).astype(np.float32)

            # Resize ke (128, 128) agar konsisten
            t_mel = torch.from_numpy(mel_db).unsqueeze(0).unsqueeze(0)
            t_tg  = torch.from_numpy(tg_db).unsqueeze(0).unsqueeze(0)
            t_mel = F.interpolate(t_mel, (128, 128), mode='bilinear', align_corners=False).squeeze(0)
            t_tg  = F.interpolate(t_tg, (128, 128), mode='bilinear', align_corners=False).squeeze(0)

            self.mels.append(t_mel)     # (1, 128, 128)
            self.tgrams.append(t_tg)    # (1, 128, 128)
            self.labels.append(lbl)
            self.mids.append(mid)

        self.mels_tensor   = torch.stack(self.mels)
        self.tgrams_tensor = torch.stack(self.tgrams)
        self.labels_tensor = torch.tensor(self.labels, dtype=torch.long)
        print(f'Selesai preload {cond}! Total: {len(self.labels)} samples di RAM.')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.mels_tensor[idx], self.tgrams_tensor[idx], self.labels_tensor[idx]

train_ds = MIMIIDataset_STgram_RAM(
    root=DATASET_ROOT, machine=MACHINE_TYPE, snr=TARGET_SNR, cond='normal',
    sr=SR, audio_len=AUDIO_LEN
)
N_CLASSES = train_ds.n_classes
train_dl  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=0, pin_memory=True, drop_last=True)
print(f'\nDataLoader: {len(train_dl)} batches/epoch | {N_CLASSES} IDs | BS={BATCH_SIZE}')


In [ ]:
# =====================================================================
# CELL 5: TRAINING — SpecAugment + LR Warmup + Cosine Annealing
# =====================================================================
model = STgramMFN_v3(n_classes=N_CLASSES, ed=EMBED_DIM).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

def get_lr(ep):
    if ep <= WARMUP_EP:
        return LR * ep / WARMUP_EP
    p = (ep - WARMUP_EP) / (EPOCHS - WARMUP_EP)
    return LR * 0.5 * (1 + math.cos(math.pi * p))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
OUT_MODEL = f'{OUT_DIR}/stgram_mfn_v3_{MACHINE_TYPE}.pt'

print(f'Model: {n_params:,} params ({n_params/1e6:.2f}M) | Device: {device}')
print(f'Training: {EPOCHS} epochs | LR_peak={LR} | Warmup={WARMUP_EP} ep')
print('=' * 65)

best_loss = float('inf')
losses = []
t0 = time.time()

for ep in range(1, EPOCHS + 1):
    lr_now = get_lr(ep)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now

    model.train()
    ep_loss = 0.0
    n_bat   = 0

    for mel, tg, lab in train_dl:
        mel = mel.to(device, non_blocking=True)
        tg  = tg.to(device, non_blocking=True)
        lab = lab.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            _, loss = model(mel, tg, lab)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        ep_loss += loss.item()
        n_bat   += 1

    avg = ep_loss / n_bat
    losses.append(avg)

    if avg < best_loss:
        best_loss = avg
        torch.save({
            'epoch': ep,
            'model_state': model.state_dict(),
            'best_loss': best_loss,
            'machine': MACHINE_TYPE,
            'n_classes': N_CLASSES,
            'embed_dim': EMBED_DIM,
            'id2label': train_ds.id2label,
            'arc_W': model.arc.class_weights().cpu(),
        }, OUT_MODEL)
        tag = ' <- BEST'
    else:
        tag = ''

    if ep % 10 == 0 or ep == 1 or ep == EPOCHS:
        print(f'Ep {ep:3d}/{EPOCHS} | Loss:{avg:.4f} | LR:{lr_now:.2e} | {(time.time()-t0)/60:.1f}min{tag}')

print(f'\nSelesai! Best loss: {best_loss:.4f} | Total Time: {(time.time()-t0)/60:.2f} min')
if torch.cuda.is_available():
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', lw=1.5, label='ArcFace Loss')
plt.axvline(x=WARMUP_EP-1, color='orange', linestyle='--', alpha=0.7, label=f'End Warmup ep{WARMUP_EP}')
plt.axhline(y=best_loss, color='green', linestyle='--', alpha=0.7, label=f'Best: {best_loss:.4f}')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Training Loss Curve — {MACHINE_TYPE.upper()} (STgram-MFN v3)', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/loss_v3_{MACHINE_TYPE}.png', dpi=150); plt.show()


In [ ]:
# =====================================================================
# CELL 6: EVALUASI PER-ID (KNN-k5, Target ArcFace, OCSVM, Ensemble)
# =====================================================================
ck = torch.load(OUT_MODEL, map_location='cpu')
eval_model = STgramMFN_v3(n_classes=ck['n_classes'], ed=ck['embed_dim']).to(device)
eval_model.load_state_dict(ck['model_state'], strict=True)
eval_model.eval()

arc_W = ck['arc_W'].numpy()  # (n_classes, embed_dim)

print('Preloading Abnormal dataset ke RAM...')
abnorm_ds = MIMIIDataset_STgram_RAM(
    DATASET_ROOT, MACHINE_TYPE, TARGET_SNR, 'abnormal',
    SR, AUDIO_LEN
)

@torch.no_grad()
def extract_all_embs(model, ds):
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
    embs = []
    for mel, tg, _ in tqdm(dl, desc='Extract Embs'):
        f = model(mel.to(device, non_blocking=True), tg.to(device, non_blocking=True))
        embs.append(f.cpu().numpy())
    return np.concatenate(embs, axis=0)

print('\nExtracting Normal Embeddings...')
norm_embs = extract_all_embs(eval_model, train_ds)
norm_labels = train_ds.labels_tensor.numpy()

print('Extracting Abnormal Embeddings...')
abnorm_embs = extract_all_embs(eval_model, abnorm_ds)
abnorm_labels = abnorm_ds.labels_tensor.numpy()

y_true = np.concatenate([np.zeros(len(norm_embs)), np.ones(len(abnorm_embs))])
all_labels = np.concatenate([norm_labels, abnorm_labels])
all_embs = np.concatenate([norm_embs, abnorm_embs], axis=0)

# 1. SCORER: KNN-Cosine Per-ID
def compute_knn_scores(k=5):
    unique_ids = np.unique(norm_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        knn = NearestNeighbors(n_neighbors=min(k, mask.sum()), metric='cosine', algorithm='brute')
        knn.fit(norm_embs[mask])
        knn_models[uid] = knn
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        dists, _ = knn_models[lid].kneighbors(all_embs[i:i+1])
        scores.append(float(dists.mean()))
    return np.array(scores)

# 2. SCORER: ArcFace Target Class Distance
def compute_arcface_target_scores():
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        w_target = arc_W[lid]
        cos_sim = np.dot(all_embs[i], w_target)
        scores.append(1.0 - cos_sim)
    return np.array(scores)

# 3. SCORER: OCSVM Per-ID
def compute_ocsvm_scores(nu=0.02):
    unique_ids = np.unique(norm_labels)
    ocsvm_models, scalers = {}, {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        sc = StandardScaler(); X = sc.fit_transform(norm_embs[mask])
        oc = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        oc.fit(X)
        ocsvm_models[uid] = oc; scalers[uid] = sc
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        X = scalers[lid].transform(all_embs[i:i+1])
        scores.append(-float(ocsvm_models[lid].decision_function(X)[0]))
    return np.array(scores)

# 4. SCORER: LOF Per-ID
def compute_lof_scores(k=15):
    unique_ids = np.unique(norm_labels)
    lof_models = {}
    for uid in unique_ids:
        mask = (norm_labels == uid)
        lof = LocalOutlierFactor(n_neighbors=min(k, mask.sum()-1), metric='cosine', novelty=True)
        lof.fit(norm_embs[mask])
        lof_models[uid] = lof
    scores = []
    for i in range(len(all_embs)):
        lid = all_labels[i]
        scores.append(-float(lof_models[lid].score_samples(all_embs[i:i+1])[0]))
    return np.array(scores)

print('Calculating Scorer Metrics...')
s_knn    = compute_knn_scores(k=5)
s_arc    = compute_arcface_target_scores()
s_ocsvm  = compute_ocsvm_scores(nu=0.02)
s_lof    = compute_lof_scores(k=15)

# Rank-normalized Ensemble
r_knn   = rankdata(s_knn) / len(s_knn)
r_arc   = rankdata(s_arc) / len(s_arc)
r_ocsvm = rankdata(s_ocsvm) / len(s_ocsvm)
r_lof   = rankdata(s_lof) / len(s_lof)
s_ens   = (r_knn * 0.4 + r_arc * 0.2 + r_ocsvm * 0.2 + r_lof * 0.2)

scorers = {
    'KNN-k5 (Per-ID)': s_knn,
    'ArcFace-Target': s_arc,
    'OCSVM (Per-ID)': s_ocsvm,
    'LOF (Per-ID)': s_lof,
    'Ensemble Scorer': s_ens
}

print('\n' + '='*55)
print(f'EVALUASI SCORERS — {MACHINE_TYPE.upper()}')
print('='*55)
best_name, best_auc, best_pauc, best_sc = '', 0, 0, None
for name, sc in scorers.items():
    auc  = roc_auc_score(y_true, sc)
    pauc = roc_auc_score(y_true, sc, max_fpr=0.1)
    print(f'{name:20s} | AUC: {auc*100:.2f}% | pAUC: {pauc*100:.2f}%')
    if auc > best_auc:
        best_auc, best_pauc, best_name, best_sc = auc, pauc, name, sc

fpr, tpr, thr = roc_curve(y_true, best_sc)
best_thr = float(thr[np.argmax(tpr - fpr)])

print('\n' + '='*55)
print(f'HASIL TERBAIK: [{best_name}]')
print(f'  AUC            : {best_auc:.4f}  ({best_auc*100:.2f}%)')
print(f'  pAUC (FPR<10%) : {best_pauc:.4f}  ({best_pauc*100:.2f}%)')
print(f'  Threshold      : {best_thr:.4f}')
print('='*55)


In [ ]:
# =====================================================================
# CELL 7: VISUALISASI FINAL
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
ax = axes[0]
ax.plot(fpr, tpr, 'b-', lw=2.5, label=f'{best_name}\nAUC={best_auc:.4f}')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
mask = fpr <= 0.1
ax.fill_between(fpr[mask], tpr[mask], alpha=0.3, color='orange', label=f'pAUC={best_pauc:.4f}')
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'ROC Curve — {MACHINE_TYPE.upper()} ({best_name})', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(); ax.grid(alpha=0.3)

# Score Distribution
ax = axes[1]
sc_n = best_sc[y_true == 0]
sc_a = best_sc[y_true == 1]
ax.hist(sc_n, bins=60, alpha=0.65, color='royalblue', label=f'Normal (n={len(sc_n)})', density=True)
ax.hist(sc_a, bins=60, alpha=0.65, color='tomato', label=f'Abnormal (n={len(sc_a)})', density=True)
ax.axvline(x=best_thr, color='green', linestyle='--', lw=2, label=f'Thr={best_thr:.3f}')
ax.set_title(f'Anomaly Score Distribution ({best_name})', fontweight='bold')
ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Density'); ax.legend(); ax.grid(alpha=0.3)

# Comparison vs Paper
ax = axes[2]
paper = {'Slider': 99.55, 'Valve': 99.64, 'Fan': 94.04, 'Pump': 91.94, f'{MACHINE_TYPE.title()} Kamu': best_auc*100}
colors = ['#aaa','#aaa','#aaa','#aaa','#2ecc71' if best_auc >= 0.9 else '#e67e22']
bars = ax.bar(paper.keys(), paper.values(), color=colors, edgecolor='white', linewidth=1.2)
for b, v in zip(bars, paper.values()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='Target 90%')
ax.set_title('Perbandingan AUC vs Paper Target', fontweight='bold')
ax.set_ylabel('AUC (%)'); ax.set_ylim(70, 105)
ax.tick_params(axis='x', rotation=20); ax.grid(alpha=0.3, axis='y'); ax.legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_v3_{MACHINE_TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Visualisasi tersimpan!')


In [ ]:
# =====================================================================
# CELL 8: EXPORT ONNX + INFERENCE CONFIG JSON (ROBUST & BEBAS ERROR)
# =====================================================================
import subprocess

# Auto-install onnx dan onnxscript jika belum tersedia di environment
try:
    import onnx
    import onnxscript
except ImportError:
    print('Menginstall onnx & onnxscript...')
    subprocess.run(['pip', 'install', '-q', 'onnx', 'onnxscript'], check=True)
    import onnx

m_cpu = STgramMFN_v3(n_classes=ck['n_classes'], ed=ck['embed_dim'])
m_cpu.load_state_dict(ck['model_state'], strict=True)
m_cpu.eval()

onnx_path = f'{OUT_DIR}/stgram_mfn_v3_{MACHINE_TYPE}.onnx'
dummy_mel = torch.randn(1, 1, 128, 128)
dummy_tg  = torch.randn(1, 1, 128, 128)

try:
    torch.onnx.export(
        m_cpu,
        (dummy_mel, dummy_tg),
        onnx_path,
        input_names=['mel', 'tgram'],
        output_names=['embedding'],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
    )
    print(f'✅ ONNX exported successfully: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')
except Exception as e:
    print(f'Fallback export: {e}')
    torch.onnx.export(
        m_cpu,
        (dummy_mel, dummy_tg),
        onnx_path,
        input_names=['mel', 'tgram'],
        output_names=['embedding'],
        opset_version=16,
        do_constant_folding=True,
        dynamic_axes={'mel': {0: 'B'}, 'tgram': {0: 'B'}, 'embedding': {0: 'B'}}
    )
    print(f'✅ ONNX exported (opset 16): {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

cfg = {
    'version': 'v3_stgram_optimized',
    'machine': MACHINE_TYPE, 'snr': TARGET_SNR,
    'best_scorer': best_name,
    'auc': float(best_auc),
    'pauc': float(best_pauc),
    'threshold': float(best_thr),
    'n_classes': int(ck['n_classes']),
    'embed_dim': int(ck['embed_dim']),
    'id2label': train_ds.id2label,
    'arc_W': arc_W.tolist(),
    'audio': {'sr': SR, 'audio_len': AUDIO_LEN},
    'features': {
        'branch_0_mel': {'n_mels': N_MELS, 'n_fft': N_FFT_MEL, 'hop_length': HOP_MEL},
        'branch_1_tgram': {'n_fft': N_FFT_TG, 'hop_length': HOP_TG, 'n_bins': N_BINS_TG}
    }
}
with open(f'{OUT_DIR}/inference_config_v3_{MACHINE_TYPE}.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print('✅ Inference config saved.')
print(f'\n🎯 SELESAI: {MACHINE_TYPE.upper()} Best AUC={best_auc:.4f} ({best_auc*100:.2f}%) | Scorer={best_name}')
